# Quickstart: Complete Pipeline in 10 Minutes

This notebook demonstrates a complete document processing pipeline:
**Ingest → Extract → Chunk → Embeddings**

## What You'll Learn
- How to use `DocpipeFlowManager` for programmatic flow execution
- Complete pipeline execution with visual outputs
- Result inspection and metrics visualization

## Prerequisites
- Ollama running on `http://localhost:11434`
- Model `nomic-embed-text` pulled: `ollama pull nomic-embed-text`
- Virtual environment activated with docpipe installed

## 1. Setup and Imports

In [ ]:
import sys
import shutil
from pathlib import Path
from pprint import pprint

# Add src to path if needed
import os
if 'PYTHONPATH' not in os.environ:
    src_path = Path.cwd().parent.parent / "src"
    sys.path.insert(0, str(src_path))

from docpipe.lib.docpipe_flow_manager import DocpipeFlowManager

print("✓ Imports loaded successfully")

## 2. Check Prerequisites

In [ ]:
import requests

def check_ollama():
    """Check if Ollama is running and has required model"""
    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=2)
        if response.status_code == 200:
            models = response.json().get('models', [])
            model_names = [m['name'] for m in models]
            if any('nomic-embed-text' in name for name in model_names):
                print("✓ Ollama is running")
                print("✓ Model 'nomic-embed-text' is available")
                return True
            print("✗ Ollama is running but 'nomic-embed-text' model not found")
            print("  Run: ollama pull nomic-embed-text")
            return False
    except Exception as e:
        print(f"✗ Ollama is not running: {e}")
        print("  Start Ollama: ollama serve")
        return False

if check_ollama():
    print("\n✓ All prerequisites met! Ready to proceed.")
else:
    print("\n✗ Please fix prerequisites before continuing.")

## 3. Create Test Data

We'll create a sample document for processing:

In [ ]:
# Create test directory
test_dir = Path("./test_data_quickstart")
test_dir.mkdir(exist_ok=True)

# Create sample document
sample_content = """
# docpipe Framework Overview

## Introduction
docpipe is a modular, operator-based data processing framework designed for building 
flexible data pipelines. It provides comprehensive operators for document processing,
extraction, chunking, and embedding generation.

## Key Features
- **Operator-Based Architecture**: 20+ specialized operators organized into categories
- **PyArrow Data Format**: Efficient memory usage and interoperability
- **DAG-Based Execution**: Directed acyclic graph workflow execution
- **Prefect Orchestration**: Parallel processing and task dependencies
- **AI/ML Integrations**: Native support for Ollama, Docling, and OpenSearch

## Use Cases
1. **Document Processing Pipelines**: Extract and process documents at scale
2. **RAG Preparation**: Prepare documents for retrieval-augmented generation
3. **Entity Extraction**: Extract structured data from unstructured documents
4. **Vector Search**: Build semantic search capabilities

## Architecture
The framework follows a hexagonal architecture pattern with clear separation between:
- Domain layer (business logic)
- Port layer (interfaces)
- Adapter layer (implementations)
- Factory layer (dependency injection)

## Getting Started
Install docling-pipelines and start processing documents in minutes. The framework provides
both CLI and programmatic APIs for maximum flexibility.
"""

sample_file = test_dir / "docpipe_overview.txt"
sample_file.write_text(sample_content)

print(f"✓ Created test directory: {test_dir}")
print(f"✓ Created sample file: {sample_file}")
print(f"  File size: {sample_file.stat().st_size} bytes")

## 4. Define Flow Configuration

Create a flow definition as a Python dictionary:

In [ ]:
flow_definition = {
    "flow_name": "quickstart-pipeline",
    "description": "Complete pipeline: Ingest → Extract → Chunk → Embeddings",
    "global_config": {
        "doc_column": "content",
        "disable_validation": True,
        "force_ingest": True
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": [str(test_dir)]},
                "include_filter": "txt,md"

            }}
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {
                "text_extraction": {
                    "provider": "docling_library",
                    "doc_column": "content"
                },
                "entity_extraction": {
                    "provider": "none"
                }
            }
        },
        {
            "name": "chunk",
            "type": "chunker",
            "depends_on": ["extract"],
            "config": {
                "chunk_type": "hybrid",
                "chunk_size": 512,
                "chunk_overlap": 128,
                "doc_column": "content"
            }
        },
        {
            "name": "embeddings",
            "type": "embeddings",
            "depends_on": ["chunk"],
            "config": {
                "provider": "litellm",
                "embeddings_column": "embeddings",
                "doc_column": "content",
                "provider_config": {
                    "model_id": "ollama/nomic-embed-text",
                    "api_base": "http://localhost:11434",
                    "api_key": "ollama"
                }
            }
        }
    ]
}

print("✓ Flow definition created")
print(f"  Flow name: {flow_definition['flow_name']}")
print(f"  Number of operators: {len(flow_definition['flow'])}")
print(f"  Pipeline: {' → '.join([op['name'] for op in flow_definition['flow']])}")

## 5. Initialize DocpipeFlowManager

In [ ]:
# Create flow manager from dictionary
manager = DocpipeFlowManager(flow_def=flow_definition)

print("✓ DocpipeFlowManager initialized")

# Display flow metadata
metadata = manager.get_execution_metadata()
print("\nFlow Metadata:")
print(f"  Flow Name: {metadata['flow_name']}")
print(f"  Description: {metadata['flow_description']}")
print(f"  Number of Operators: {metadata['num_operators']}")
print(f"  Job ID: {metadata['job_id']}")
print(f"  Job Run ID: {metadata['job_run_id']}")

## 6. Execute the Pipeline

This will process the document through all operators:

In [ ]:
print("Executing pipeline...")
print("Pipeline: Ingest → Extract → Chunk → Embeddings")
print("This may take a few moments...\n")

try:
    result = manager.execute()
    print("\n✓ Pipeline executed successfully!")
except Exception as e:
    print(f"\n✗ Pipeline execution failed: {e}")
    raise

## 7. Inspect Results

Check the execution status and results:

In [ ]:
# Get execution status
if manager.orchestrator:
    job_stats_service = getattr(manager.orchestrator, "job_stats_service", None)
    if job_stats_service:
        try:
            job_stats = job_stats_service.get_job_run_stats(job_run_id=manager.job_run_id)
            if job_stats and hasattr(job_stats, "status"):
                print(f"Execution Status: {job_stats.status}")
                print(f"Job Run ID: {manager.job_run_id}")
                
                if job_stats.status == "Completed":
                    print("\n✓ All operators completed successfully!")
                elif job_stats.status == "Failed":
                    print("\n✗ Execution failed. Check logs for details.")
        except Exception as e:
            print(f"Could not retrieve job status: {e}")
else:
    print("Orchestrator not available for status check")

## 8. View Execution Logs

Display the last few log lines:

In [ ]:
logs = manager.get_execution_logs()
print(f"Total log lines captured: {len(logs)}")

if logs:
    print("\nLast 15 log lines:")
    print("=" * 80)
    for line in logs[-15:]:
        print(line)
else:
    print("No logs captured")

## 9. Cleanup

Remove test data:

In [ ]:
if test_dir.exists():
    shutil.rmtree(test_dir)
    print(f"✓ Cleaned up test directory: {test_dir}")
else:
    print("Test directory already removed")

## Summary

You've successfully:
1. ✓ Set up the environment and checked prerequisites
2. ✓ Created test data
3. ✓ Defined a flow configuration
4. ✓ Initialized DocpipeFlowManager
5. ✓ Executed a complete pipeline (Ingest → Extract → Chunk → Embeddings)
6. ✓ Inspected results and logs
7. ✓ Cleaned up test data

## Next Steps

- **[02_operator_showcase.ipynb](02_operator_showcase.ipynb)** - Explore all available operators
- **[03_document_extraction.ipynb](03_document_extraction.ipynb)** - Deep dive into document extraction
- **[04_embeddings_vectordb.ipynb](04_embeddings_vectordb.ipynb)** - Vector storage and retrieval
- **Modify the flow** - Try different operators, parameters, or document types
- **Check documentation** - See [README.md](../../README.md) for more information